In [ ]:
import json
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack
import warnings
from gensim.models import Word2Vec
warnings.filterwarnings('ignore')

print("SVM Fake News Detector - Simple Model")
print("="*60)

In [ ]:
DATA_PATH = "../data/raw/multinli_1.0/multinli_1.0_train.jsonl"

data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["gold_label"] != "-":
            data.append(item)

data = data

df = pd.DataFrame(data)[["sentence1", "sentence2", "gold_label"]]

print(f"Dataset: {len(df)} samples")
print("\nLabel distribution:")
print(df["gold_label"].value_counts())

Dataset: 100000 samples

Label distribution:
gold_label
contradiction    35140
entailment       33803
neutral          31057
Name: count, dtype: int64


In [ ]:
label_mapping = {"entailment": 0, "neutral": 1, "contradiction": 2}

sentence1_texts = df["sentence1"].values
sentence2_texts = df["sentence2"].values
labels = df["gold_label"].map(label_mapping).values

print(f"Sentence1 shape: {sentence1_texts.shape}")
print(f"Sentence2 shape: {sentence2_texts.shape}")
print(f"Labels shape: {labels.shape}")
print(f"\nExample sentence1:\n{sentence1_texts[0][:100]}...")
print(f"Example sentence2:\n{sentence2_texts[0][:100]}...")

Sentence1 shape: (100000,)
Sentence2 shape: (100000,)
Labels shape: (100000,)

Example sentence1:
Conceptually cream skimming has two basic dimensions - product and geography....
Example sentence2:
Product and geography are what make cream skimming work. ...


In [ ]:
# Add advanced text features for better accuracy
def extract_advanced_features(s1_array, s2_array):
    features = []
    
    for s1, s2 in zip(s1_array, s2_array):
        # Length ratios
        len1, len2 = len(s1.split()), len(s2.split())
        len_ratio = min(len1, len2) / max(len1, len2) if max(len1, len2) > 0 else 0
        
        # Character-level features
        char_ratio = min(len(s1), len(s2)) / max(len(s1), len(s2)) if max(len(s1), len(s2)) > 0 else 0
        
        # Negation indicators
        negations = ['not', 'no', 'never', 'neither', 'none', 'nobody', "n't"]
        neg1 = sum(1 for word in s1.lower().split() if word in negations)
        neg2 = sum(1 for word in s2.lower().split() if word in negations)
        neg_diff = abs(neg1 - neg2)
        
        # Common words ratio (excluding stopwords)
        words1 = set(s1.lower().split())
        words2 = set(s2.lower().split())
        common_words = len(words1 & words2)
        total_unique = len(words1 | words2)
        common_ratio = common_words / total_unique if total_unique > 0 else 0
        
        features.append([len_ratio, char_ratio, neg_diff, common_ratio, neg1, neg2])
    
    return np.array(features)

advanced_feats = extract_advanced_features(sentence1_texts, sentence2_texts)
print(f"Advanced features shape: {advanced_feats.shape}")

Advanced features shape: (100000, 6)


In [ ]:
print("New Feature set...")

# Separate word-level TF-IDF for each sentence
word_vectorizer = TfidfVectorizer(
    max_features=10000,
    max_df=0.85,
    min_df=3,
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm='l2',
    analyzer='word'
)

X1_word = word_vectorizer.fit_transform(sentence1_texts)
X2_word = word_vectorizer.transform(sentence2_texts)


# Interaction features
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

print("Computing interaction features...")
# Word-level cosine similarity
cosine_sim_word = np.array([
    cosine_similarity(X1_word[i], X2_word[i])[0][0] 
    for i in range(X1_word.shape[0])
]).reshape(-1, 1)


# Element-wise operations on word vectors
X_diff = X1_word - X2_word
X_mult = X1_word.multiply(X2_word)

# Enhanced basic features
def enhanced_features(s1, s2):
    words1 = s1.lower().split()
    words2 = s2.lower().split()
    set1, set2 = set(words1), set(words2)
    
    # Jaccard similarity
    jaccard = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0
    
    # Length features
    len1, len2 = len(words1), len(words2)
    len_ratio = min(len1, len2) / max(len1, len2) if max(len1, len2) > 0 else 0
    
    # Negation words
    negations = {'not', 'no', 'never', 'neither', 'none', 'nobody', "n't", 'nothing', 'nowhere'}
    neg1 = sum(1 for w in words1 if w in negations)
    neg2 = sum(1 for w in words2 if w in negations)
    
    return [jaccard, len1, len2, len_ratio, abs(len1-len2), neg1, neg2, abs(neg1-neg2)]

print("Extracting enhanced features...")
enhanced_feats = np.array([enhanced_features(s1, s2) for s1, s2 in zip(sentence1_texts, sentence2_texts)])

# Normalize numerical features
scaler = StandardScaler()
enhanced_feats_scaled = scaler.fit_transform(enhanced_feats)
advanced_feats_scaled = scaler.fit_transform(advanced_feats)

# Combine ALL features - each sentence separately
X = hstack([
    X1_word,                    # Sentence1 word TF-IDF
    X2_word,                    # Sentence2 word TF-IDF
    X_diff,                     # Difference (word-level)
    X_mult,                     # Multiplication (word-level)
    cosine_sim_word,            # Word cosine similarity
    enhanced_feats_scaled,      # Scaled enhanced features
    advanced_feats_scaled       # Scaled advanced features
])

print(f"✓ Final feature shape: {X.shape}")
print(f"  - Word TF-IDF: {X1_word.shape[1] * 2}")
print(f"  - Interactions: {X1_word.shape[1] + 2}")
print(f"  - Enhanced: {enhanced_feats_scaled.shape[1] + advanced_feats_scaled.shape[1]}")

New Feature set...
Computing interaction features...
Extracting enhanced features...
✓ Final feature shape: (100000, 40015)
  - Word TF-IDF: 20000
  - Interactions: 10002
  - Enhanced: 14


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

Train samples: 80000
Test samples: 20000
Features: 40015


In [ ]:
print("Training Random Forest with tuned hyperparameters...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=50,
    min_samples_split=20,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    random_state=43,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train, y_train)
print("✓ Training complete")

Training Random Forest with tuned hyperparameters...
✓ Training complete


In [ ]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
target_names = ["Entailment (VRAI)", "Neutral (À_VÉRIFIER)", "Contradiction (FAUX)"]
print(classification_report(y_test, y_pred, target_names=target_names))

Test Accuracy: 0.5588
Cohen's Kappa: 0.3404

CLASSIFICATION REPORT
                      precision    recall  f1-score   support

   Entailment (VRAI)       0.54      0.64      0.59      6761
Neutral (À_VÉRIFIER)       0.50      0.60      0.55      6211
Contradiction (FAUX)       0.67      0.45      0.54      7028

            accuracy                           0.56     20000
           macro avg       0.57      0.56      0.56     20000
        weighted avg       0.58      0.56      0.56     20000



In [ ]:
print("Applying TruncatedSVD (PCA for sparse matrices) for dimensionality reduction...")
print(f"Original feature dimension: {X_train.shape[1]}")

# Apply TruncatedSVD to reduce dimensionality
n_components = 500  # Reduce to 500 dimensions (adjust as needed)
svd = TruncatedSVD(n_components=n_components, random_state=42)

X_train_reduced = svd.fit_transform(X_train)
X_test_reduced = svd.transform(X_test)

explained_variance = svd.explained_variance_ratio_.sum()

print(f"Reduced feature dimension: {X_train_reduced.shape[1]}")
print(f"Explained variance: {explained_variance:.4f} ({explained_variance*100:.2f}%)")
print("✓ Dimensionality reduction complete")

In [ ]:
print("Training LinearSVC with optimized parameters on reduced features...")

# LinearSVC performs better with higher C for this task
svm_model = LinearSVC(
    C=1.0,                    # Increased regularization parameter
    max_iter=2000,            # More iterations
    random_state=42,
    dual=False,               # Use primal formulation (faster for n_samples > n_features)
    class_weight='balanced',  # Handle class imbalance
    loss='squared_hinge'      # Standard SVM loss
)

svm_model.fit(X_train_reduced, y_train)
print("✓ Training complete")

Training LinearSVC with optimized parameters...
✓ Training complete


In [ ]:
y_pred = svm_model.predict(X_test_reduced)
accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
target_names = ["Entailment (VRAI)", "Neutral (À_VÉRIFIER)", "Contradiction (FAUX)"]
print(classification_report(y_test, y_pred, target_names=target_names))

Test Accuracy: 0.5188
Cohen's Kappa: 0.2774

CLASSIFICATION REPORT
                      precision    recall  f1-score   support

   Entailment (VRAI)       0.51      0.54      0.52      6761
Neutral (À_VÉRIFIER)       0.49      0.49      0.49      6211
Contradiction (FAUX)       0.55      0.53      0.54      7028

            accuracy                           0.52     20000
           macro avg       0.52      0.52      0.52     20000
        weighted avg       0.52      0.52      0.52     20000



In [ ]:
# Create a 'models' directory if it doesn't exist

models_dir = 'models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created directory: {models_dir}")

# Save the Random Forest model
rf_model_filename = os.path.join(models_dir, 'random_forest_model.joblib')
joblib.dump(rf_model, rf_model_filename)
print(f"Random Forest model saved to {rf_model_filename}")

# Save the TruncatedSVD transformer
svd_filename = os.path.join(models_dir, 'svd_transformer.joblib')
joblib.dump(svd, svd_filename)
print(f"TruncatedSVD transformer saved to {svd_filename}")

# Save the LinearSVC model
svm_model_filename = os.path.join(models_dir, 'linear_svc_model.joblib')
joblib.dump(svm_model, svm_model_filename)
print(f"LinearSVC model saved to {svm_model_filename}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

for i, label in enumerate(["Entailment", "Neutral", "Contradiction"]):
    class_acc = cm[i, i] / cm[i].sum()
    print(f"{label}: {class_acc:.4f}")

Confusion Matrix:
[[3625 1617 1519]
 [1698 3019 1494]
 [1749 1547 3732]]
Entailment: 0.5362
Neutral: 0.4861
Contradiction: 0.5310
